## Package imports

In [2]:
import numpy as np

from sensors.phone_IMU import phone_IMU

from robots.robot_state import phone_state
from robots.phone import phone

from filters.filter_state import phone_filter_state
from filters.filter_covariance import filter_covariance
from filters.Attitude_observer import Attitude_observer

from simulations.phone_sim import phone_sim

from util import pose_utils, quat_utils, phone_utils

## Environment definition

In [3]:
# parameters for operational area
g = -9.81*np.array([[0.],[0.],[1.]])            # gravity vector
m_az = np.deg2rad(1.33)                         # magnetic field azimuth Zaragoza
m_el = np.deg2rad(56.82)                        # magnetic field elevation Zaragoza
m = np.array([[np.cos(m_el)*np.cos(m_az)],      # magnetic field vector
              [np.cos(m_el)*np.sin(m_az)],
              [np.sin(m_el)]])

# consider replacing m with your own initial measurement
# m = np.array([[0.0], [0.0], [0.0]])

## Robot configuration

In [4]:
# we are using the sensor data read from a data file in two ways:
# - to directly reconstruct the phone attitude, presented as "ground truth" for the phone
# - to provide sensor measurements from an IMU sensor that can be provided to a filter/observer
gyr, acc, mag = phone_utils.read_sensorLog('data/sensorLog.txt')

In [5]:
# configure phone phone_1

# parameters for phone phone_1
phone_1_params = phone.parameters()
phone_1_params.g = g        # gravity vector
phone_1_params.gyr = gyr    # gyroscope measurements
phone_1_params.acc = acc    # normalized accelerometer measurements
phone_1_params.mag = mag    # normalized magnetometer measurements

# configuration flags for phone phone_1
phone_1_config = phone.configuration()

## Sensor configuration

In [6]:
# configure sensors

# IMU (3-axis gyroscope, 3-axis accelerometer, 3-axis magnetometer)
IMU_params = phone_IMU.parameters()
IMU_params.freq = 100.0                 # measurement frequency of accelerometer and gyroscope in Hz (MUST be an integer multiple of freq_m)

IMU_params.gyr = gyr                    # gyroscope measurements
IMU_params.acc = acc                    # normalized accelerometer measurements
IMU_params.mag = mag                    # normalized magnetometer measurements

## Filter configuration

In [7]:
# Observer configuration

Obs_params = Attitude_observer.parameters()

Obs_params.k_a = 20.0       # gain for accelerometer measurements
Obs_params.k_m = 2.0        # gain for magnetometer measurements

Obs_params.g = g           # estimate of gravity vector
Obs_params.m = m           # estimate of magnetic field vector

# observer type
Obs_config = Attitude_observer.configuration()

## Robot definition

In [8]:
# phone phone_1
phone_1_name = 'phone_1'
phone_1_color = 'rgb(0,0,0)'

# sensors
phone_1_IMU_sensor = phone_IMU(IMU_params)
phone_1_IMU_sensor.set_runtime_attributes('IMU', phone_1_color, {})

phone_1_sensors = (phone_1_IMU_sensor,)

# observers
phone_1_observer = Attitude_observer(Obs_params, Obs_config)
name = 'Observer'
color = 'rgb(0,0,255)'
phone_1_observer.set_runtime_attributes(name, color, {})

phone_1_filters = (phone_1_observer,)

# phone
phone_1 = phone(phone_1_params, phone_1_config, phone_1_sensors, phone_1_filters)
phone_1.set_runtime_attributes(phone_1_name, phone_1_color, {})

## Robot and sensor testing

In [9]:
# # initial state for phone phone_1
# phone_1_q0 = np.array([[1.], [0.], [0.], [0.]])
# phone_1_eul0 = np.array([[0.],[0.],[0.]])

# phone_1_s0 = phone_state(phone_1_q0, phone_1_eul0)

# # precompute phone trajectory (direct reconstruction) and all measurements
# dt = 0.01
# phone_1.set_dt(dt)
# phone_1.precompute_trajectory(phone_1_s0)
# m_traj, m_true_traj, m_e_traj = phone_1.measure_all()

In [10]:
# from plots.plotly_plots import plotter

# plt = plotter()
# plt.plot_robot_state(phone_1, 'eul', dt)
# #plt.plot_measurement(phone_1, 'w', dt, m_traj, m_true_traj, m_e_traj)
# #plt.plot_measurement(phone_1, 'y_a', dt, m_traj, m_true_traj, m_e_traj)
# #plt.plot_measurement(phone_1, 'y_m', dt, m_traj, m_true_traj, m_e_traj)

## Simulation configuration

In [11]:
# initial state for phone phone_1
phone_1_q0 = np.array([[1.], [0.], [0.], [0.]])
phone_1_eul0 = np.array([[0.],[0.],[0.]])

phone_1_s0 = phone_state(phone_1_q0, phone_1_eul0)

In [12]:
# robot data for simulation: list of pairs of the form (robot, initial state)
robot_data = [
    (phone_1, phone_1_s0)
]

In [13]:
# configure simulation parameters 
sim_params = phone_sim.parameters()

sim_params.freq = 100.0                 # fixed simulation frequency (in Hz)

# configuration flags for simulation
sim_config = phone_sim.configuration()

sim_config.precompute_robot_trajectories = True

sim_config.record_inival = True        # this can be overridden by argument enable_replay to sim_run()
sim_config.record_measurement = True   # this can be overridden by argument enable_replay to sim_run()

## Simulation and filter testing

In [16]:
# initialize simulation
sim = phone_sim(sim_params, sim_config, robot_data)

# run filter simulation (single run)
result_data = sim.sim_run(0, enable_replay = True)
replay_data = sim.replay(result_data, record_pos_error_ellipsoid = True)

In [15]:
from plots.plotly_plots import plotter

plt = plotter()
plt.plot_filter_state(phone_1, 'eul', sim.dt, replay_data, 0)

## Monte Carlo filter simulation 

In [ ]:
# more package imports
import multiprocessing as mp
from functools import partial
import tqdm as tqdm
import shelve

In [ ]:
# Monte Carlo simulation

# initialize simulation
sim = phone_sim(sim_params, sim_config, robot_data)

def run(sim: phone_sim, result_data_file: str, enable_replay: bool, 
        n_MC: int, n_proc: int, n_reuse: int):

    # open data shelf, OVERWRITES PREVIOUS FILE!!!
    data = shelve.open(result_data_file, flag = 'n')

    # shelve any global data needed for replaying/debugging
    data['sim'] = sim
    data['n_MC'] = n_MC
    
    # parallelized simulation loop
    with mp.Pool(n_proc, maxtasksperchild = n_reuse) as p, tqdm.tqdm(total=n_MC) as pbar:
        for result_data in p.imap_unordered(partial(sim.sim_run, enable_replay = enable_replay), range(n_MC)):
            # update the progress bar
            pbar.update()
            pbar.refresh()
                    
            # shelve the result data
            run_id = result_data['run_id']
            data['result_data[{}]'.format(run_id)] = result_data

    # close the data shelf
    data.close()

# CONFIGURE HERE

result_data_file = 'data/result_data'

# set enable_replay to true to record filter trajectories
enable_replay = True

# number of Monte Carlo runs
n_MC = 1

# number of parallel processes
n_proc = 16

# max times a child process can be reused before resetting (stops ballooning memory consumption)
n_reuse = 5

run(sim, result_data_file, enable_replay, n_MC, n_proc, n_reuse)

## Interactive app

In [1]:
from UI.simple_app import simple_app

# result data file, change this to read different saved/renamed result data files
data_file = 'data/result_data'

# where should the app buffer the downloadable file
downloadable_file_buffer = 'data/single_run'

# create the app
dash_app = simple_app(data_file, downloadable_file_buffer)

# start app and display in external browser tab
dash_app.app.run(jupyter_mode = 'external')

Dash app running on http://127.0.0.1:8050/
